# Enriquecimiento y Unificación

**Entrada:** `RAW.TRIPS_YELLOW`, `RAW.TRIPS_GREEN` + descarga taxi zone lookups  
**Salida:** `ANALYTICS.INT_TRIPS_ENRICHED`

Hace:
- Carga los 4 lookups a RAW (taxi zones, payment types, rate codes, vendors)
- Lee raw chunk por chunk (mes a mes)
- Homogeneiza esquemas yellow/green
- Une yellow + green
- Enriquece con zonas y catálogos
- Escribe tabla intermedia particionada por año/mes

## Imports y configuración

In [13]:
import os
import time
import urllib.request
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import snowflake.connector

SNOWFLAKE_ACCOUNT = os.environ["SNOWFLAKE_ACCOUNT"]
SNOWFLAKE_USER = os.environ["SNOWFLAKE_USER"]
SNOWFLAKE_PASSWORD = os.environ["SNOWFLAKE_PASSWORD"]
SNOWFLAKE_DATABASE = os.environ["SNOWFLAKE_DATABASE"]
SNOWFLAKE_WAREHOUSE = os.environ["SNOWFLAKE_WAREHOUSE"]
SNOWFLAKE_ROLE = os.environ.get("SNOWFLAKE_ROLE", "SYSADMIN")
SCHEMA_RAW = os.environ.get("SNOWFLAKE_SCHEMA_RAW", "RAW")
SCHEMA_ANALYTICS = os.environ.get("SNOWFLAKE_SCHEMA_ANALYTICS", "ANALYTICS")  

START_YEAR = int(os.environ.get("START_YEAR", "2015"))  
END_YEAR = int(os.environ.get("END_YEAR",   "2025"))  
RUN_ID = os.environ.get("RUN_ID", "run_001")        

spark = SparkSession.builder \
    .appName("P3_Enriquecimiento") \
    .config("spark.jars.packages",
            "net.snowflake:spark-snowflake_2.12:2.16.0-spark_3.4,"
            "net.snowflake:snowflake-jdbc:3.16.1") \
    .getOrCreate()

sf_raw = {
    "sfURL"      : f"{SNOWFLAKE_ACCOUNT}.snowflakecomputing.com",
    "sfUser"     : SNOWFLAKE_USER,
    "sfPassword" : SNOWFLAKE_PASSWORD,
    "sfDatabase" : SNOWFLAKE_DATABASE,
    "sfSchema"   : SCHEMA_RAW,
    "sfWarehouse": SNOWFLAKE_WAREHOUSE,
    "sfRole"     : SNOWFLAKE_ROLE,
}

sf_analytics = {  
    **sf_raw,
    "sfSchema": SCHEMA_ANALYTICS,
}

print(f"Spark {spark.version} | {SNOWFLAKE_DATABASE}.{SCHEMA_RAW} → {SCHEMA_ANALYTICS}")
print(f"Rango: {START_YEAR}–{END_YEAR} | RUN_ID: {RUN_ID}")

Spark 3.5.0 | NYC_TLC.RAW → ANALYTICS
Rango: 2015–2025 | RUN_ID: run_001


## Verificación de datos 
Es un diagnóstico inicial que verifica qué vendor IDs, payment types y rate codes existen en los datos antes de crear los catálogos.

Se realizó ya que se encontraron datos diferentes que en la documentación en el preview de datos en Snowflake.

In [25]:
print("Diagnóstico inicial")

conn = snowflake.connector.connect(
    account=SNOWFLAKE_ACCOUNT, user=SNOWFLAKE_USER,
    password=SNOWFLAKE_PASSWORD, database=SNOWFLAKE_DATABASE,
    schema=SCHEMA_RAW, warehouse=SNOWFLAKE_WAREHOUSE, role=SNOWFLAKE_ROLE,
)
cursor = conn.cursor()

# Vendors 
cursor.execute(f"""
    SELECT VENDORID, COUNT(*) as TRIPS
    FROM {SCHEMA_RAW}.TRIPS_YELLOW
    GROUP BY VENDORID
    UNION ALL
    SELECT VENDORID, COUNT(*) as TRIPS
    FROM {SCHEMA_RAW}.TRIPS_GREEN
    GROUP BY VENDORID
    ORDER BY VENDORID
""")
print("\nVendor IDs en los datos:")
for row in cursor:
    print(f"  VendorID {row[0]}: {row[1]:,} viajes")

# Payment types 
cursor.execute(f"""
    SELECT PAYMENT_TYPE, COUNT(*) as TRIPS
    FROM {SCHEMA_RAW}.TRIPS_YELLOW
    GROUP BY PAYMENT_TYPE
    UNION ALL
    SELECT PAYMENT_TYPE, COUNT(*) as TRIPS
    FROM {SCHEMA_RAW}.TRIPS_GREEN
    GROUP BY PAYMENT_TYPE
    ORDER BY PAYMENT_TYPE
""")
print("\nPayment Types en los datos:")
for row in cursor:
    print(f"  PaymentType {row[0]}: {row[1]:,} viajes")

# Rate codes
cursor.execute(f"""
    SELECT RATECODEID, COUNT(*) as TRIPS
    FROM {SCHEMA_RAW}.TRIPS_YELLOW
    GROUP BY RATECODEID
    UNION ALL
    SELECT RATECODEID, COUNT(*) as TRIPS
    FROM {SCHEMA_RAW}.TRIPS_GREEN
    GROUP BY RATECODEID
    ORDER BY RATECODEID
""")
print("\nRate Code IDs en los datos:")
for row in cursor:
    print(f"  RateCodeID {row[0]}: {row[1]:,} viajes")

# Trip types (solo para green) 
cursor.execute(f"""
    SELECT TRIP_TYPE, COUNT(*) as TRIPS
    FROM {SCHEMA_RAW}.TRIPS_GREEN
    GROUP BY TRIP_TYPE
    ORDER BY TRIP_TYPE
""")
print("\nTrip Types en datos Green:")
for row in cursor:
    print(f"  TripType {row[0]}: {row[1]:,} viajes")

conn.close()
print("\n Diagnóstico completo")


Diagnóstico inicial

Vendor IDs en los datos:
  VendorID 1: 312,908,376 viajes
  VendorID 1: 12,935,843 viajes
  VendorID 2: 487,075,187 viajes
  VendorID 2: 55,275,879 viajes
  VendorID 3: 10,297 viajes
  VendorID 4: 758,881 viajes
  VendorID 5: 1,079 viajes
  VendorID 5: 47 viajes
  VendorID 6: 263,289 viajes
  VendorID 6: 27,285 viajes
  VendorID 7: 536,131 viajes

Payment Types en los datos:
  PaymentType 0: 21,172,274 viajes
  PaymentType 1: 547,370,832 viajes
  PaymentType 1: 33,993,339 viajes
  PaymentType 2: 225,343,518 viajes
  PaymentType 2: 31,839,310 viajes
  PaymentType 3: 3,965,779 viajes
  PaymentType 3: 290,453 viajes
  PaymentType 4: 3,700,730 viajes
  PaymentType 4: 177,985 viajes
  PaymentType 5: 107 viajes
  PaymentType 5: 2,941 viajes
  PaymentType None: 1,935,026 viajes

Rate Code IDs en los datos:
  RateCodeID 1: 752,586,976 viajes
  RateCodeID 1: 64,357,590 viajes
  RateCodeID 2: 19,934,786 viajes
  RateCodeID 2: 164,232 viajes
  RateCodeID 3: 1,760,665 viajes
 

## Taxi Zone Lookup y Catalogos

Descarga el CSV oficial de NYC TLC (~12 KB, 265 zonas) y lo sube a RAW. También crea tres catalogos (Payment Type, Rate Code, Vendor) con tablas estaticas basadas en el diccionario oficial de NYC TLC.



In [26]:
print("Cargando lookups a raw...")
conn = snowflake.connector.connect(
    account=SNOWFLAKE_ACCOUNT, user=SNOWFLAKE_USER,
    password=SNOWFLAKE_PASSWORD, database=SNOWFLAKE_DATABASE,
    schema=SCHEMA_RAW, warehouse=SNOWFLAKE_WAREHOUSE, role=SNOWFLAKE_ROLE,
)
cursor = conn.cursor()

# Taxi Zones 
import urllib.request
TAXI_ZONES_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
local_csv = "/tmp/taxi_zone_lookup.csv"
urllib.request.urlretrieve(TAXI_ZONES_URL, local_csv)
df_zones = spark.read.option("header", True).option("inferSchema", True).csv(local_csv)
df_zones = df_zones.toDF(*[c.lower() for c in df_zones.columns])
df_zones.write.format("net.snowflake.spark.snowflake") \
    .options(**sf_raw).option("dbtable", "TAXI_ZONES").mode("overwrite").save()
print(f"  TAXI_ZONES: {df_zones.count()} filas")

# Vendors - los unknown aparecieron en verificación de datos
cursor.execute(f"""
    CREATE OR REPLACE TABLE {SCHEMA_RAW}.DIM_VENDOR (
        VENDOR_ID   INTEGER,
        VENDOR_NAME VARCHAR
    )
""")
cursor.executemany(
    f"INSERT INTO {SCHEMA_RAW}.DIM_VENDOR VALUES (%s, %s)",
    [
        (1, "Creative Mobile Technologies, LLC"),
        (2, "Curb Mobility, LLC"),
        (3, "Unknown"),
        (4, "Unknown"),
        (5, "Unknown"),
        (6, "Myle Technologies Inc"),
        (7, "Helix"),
    ]
)
print(f"  DIM_VENDOR: 7 filas")

# Payment Types 
cursor.execute(f"""
    CREATE OR REPLACE TABLE {SCHEMA_RAW}.DIM_PAYMENT_TYPE (
        PAYMENT_TYPE_ID   INTEGER,
        PAYMENT_TYPE_DESC VARCHAR
    )
""")
cursor.executemany(
    f"INSERT INTO {SCHEMA_RAW}.DIM_PAYMENT_TYPE VALUES (%s, %s)",
    [
        (0, "Flex Fare trip"),
        (1, "Credit card"),
        (2, "Cash"),
        (3, "No charge"),
        (4, "Dispute"),
        (5, "Unknown"),
        (6, "Voided trip"),
        (None, "Not specified"),  
    ]
)
print(f"  DIM_PAYMENT_TYPE: 8 filas")

# Rate Codes 
cursor.execute(f"""
    CREATE OR REPLACE TABLE {SCHEMA_RAW}.DIM_RATE_CODE (
        RATE_CODE_ID   INTEGER,
        RATE_CODE_DESC VARCHAR
    )
""")
cursor.executemany(
    f"INSERT INTO {SCHEMA_RAW}.DIM_RATE_CODE VALUES (%s, %s)",
    [
        (1, "Standard rate"),
        (2, "JFK"),
        (3, "Newark"),
        (4, "Nassau or Westchester"),
        (5, "Negotiated fare"),
        (6, "Group ride"),
        (99, "Null/unknown"),
        (None, "Not specified"),  
    ]
)
print(f"  DIM_RATE_CODE: 8 filas")

# Trip Type que solo tiene green
cursor.execute(f"""
    CREATE OR REPLACE TABLE {SCHEMA_RAW}.DIM_TRIP_TYPE (
        TRIP_TYPE_ID   INTEGER,
        TRIP_TYPE_DESC VARCHAR
    )
""")
cursor.executemany(
    f"INSERT INTO {SCHEMA_RAW}.DIM_TRIP_TYPE VALUES (%s, %s)",
    [
        (1, "Street-hail"),
        (2, "Dispatch"),
        (None, "Not specified"),  
    ]
)
print(f"  DIM_TRIP_TYPE: 3 filas")

conn.commit()
conn.close()
print(f"\nTodos los lookups cargados en {SCHEMA_RAW}.")

Cargando lookups a raw...
  TAXI_ZONES: 265 filas
  DIM_VENDOR: 7 filas
  DIM_PAYMENT_TYPE: 8 filas
  DIM_RATE_CODE: 8 filas
  DIM_TRIP_TYPE: 3 filas

Todos los lookups cargados en RAW.


## Creacion de INT_TRIPS_ENRICHED 

Usamos SQL para hacer la tabla INT_TRIPS_ENRICHED en Snowflake raw, con todos los datos ya unidos y enriquecidos.

In [27]:
print("Creando INT_TRIPS_ENRICHED...")
conn = snowflake.connector.connect(
    account=SNOWFLAKE_ACCOUNT, user=SNOWFLAKE_USER,
    password=SNOWFLAKE_PASSWORD, database=SNOWFLAKE_DATABASE,
    schema=SCHEMA_RAW, warehouse=SNOWFLAKE_WAREHOUSE, role=SNOWFLAKE_ROLE,
)
cursor = conn.cursor()
sql = f"""
CREATE OR REPLACE TABLE {SNOWFLAKE_DATABASE}.{SCHEMA_RAW}.INT_TRIPS_ENRICHED AS
-- Yellow
SELECT
    'yellow' AS SERVICE_TYPE,
    VENDORID AS VENDOR_ID,
    PICKUP_DATETIME,
    DROPOFF_DATETIME,
    PASSENGER_COUNT,
    TRIP_DISTANCE,
    RATECODEID AS RATE_CODE_ID,
    STORE_AND_FWD_FLAG,
    PULOCATIONID AS PU_LOCATION_ID,
    DOLOCATIONID AS DO_LOCATION_ID,
    PAYMENT_TYPE,
    FARE_AMOUNT,
    EXTRA,
    MTA_TAX,
    TIP_AMOUNT,
    TOLLS_AMOUNT,
    IMPROVEMENT_SURCHARGE,
    TOTAL_AMOUNT,
    CONGESTION_SURCHARGE,
    AIRPORT_FEE,
    NULL AS EHAIL_FEE,
    NULL AS TRIP_TYPE,
    NULL AS TRIP_TYPE_DESC,
    -- Zonas
    tz_pu.BOROUGH AS PU_BOROUGH,
    tz_pu.ZONE AS PU_ZONE,
    tz_do.BOROUGH AS DO_BOROUGH,
    tz_do.ZONE AS DO_ZONE,
    -- Catálogos
    v.VENDOR_NAME,
    pt.PAYMENT_TYPE_DESC,
    rc.RATE_CODE_DESC,
    -- Lineage
    RUN_ID,
    SOURCE_YEAR,
    SOURCE_MONTH,
    SOURCE_PATH,
    INGESTED_AT_UTC
FROM {SNOWFLAKE_DATABASE}.{SCHEMA_RAW}.TRIPS_YELLOW
LEFT JOIN {SNOWFLAKE_DATABASE}.{SCHEMA_RAW}.TAXI_ZONES tz_pu ON PULOCATIONID = tz_pu.LOCATIONID
LEFT JOIN {SNOWFLAKE_DATABASE}.{SCHEMA_RAW}.TAXI_ZONES tz_do ON DOLOCATIONID = tz_do.LOCATIONID
LEFT JOIN {SNOWFLAKE_DATABASE}.{SCHEMA_RAW}.DIM_VENDOR v ON VENDORID = v.VENDOR_ID
LEFT JOIN {SNOWFLAKE_DATABASE}.{SCHEMA_RAW}.DIM_PAYMENT_TYPE pt ON PAYMENT_TYPE = pt.PAYMENT_TYPE_ID
LEFT JOIN {SNOWFLAKE_DATABASE}.{SCHEMA_RAW}.DIM_RATE_CODE rc ON RATECODEID = rc.RATE_CODE_ID
UNION ALL
-- Green
SELECT
    'green' AS SERVICE_TYPE,
    VENDORID AS VENDOR_ID,
    PICKUP_DATETIME,
    DROPOFF_DATETIME,
    PASSENGER_COUNT,
    TRIP_DISTANCE,
    RATECODEID AS RATE_CODE_ID,
    STORE_AND_FWD_FLAG,
    PULOCATIONID AS PU_LOCATION_ID,
    DOLOCATIONID AS DO_LOCATION_ID,
    PAYMENT_TYPE,
    FARE_AMOUNT,
    EXTRA,
    MTA_TAX,
    TIP_AMOUNT,
    TOLLS_AMOUNT,
    IMPROVEMENT_SURCHARGE,
    TOTAL_AMOUNT,
    CONGESTION_SURCHARGE,
    NULL AS AIRPORT_FEE,
    EHAIL_FEE,
    TRIP_TYPE,
    tt.TRIP_TYPE_DESC,
    -- Zonas
    tz_pu.BOROUGH AS PU_BOROUGH,
    tz_pu.ZONE AS PU_ZONE,
    tz_do.BOROUGH AS DO_BOROUGH,
    tz_do.ZONE AS DO_ZONE,
    -- Catálogos
    v.VENDOR_NAME,
    pt.PAYMENT_TYPE_DESC,
    rc.RATE_CODE_DESC,
    -- Lineage
    RUN_ID,
    SOURCE_YEAR,
    SOURCE_MONTH,
    SOURCE_PATH,
    INGESTED_AT_UTC
FROM {SNOWFLAKE_DATABASE}.{SCHEMA_RAW}.TRIPS_GREEN
LEFT JOIN {SNOWFLAKE_DATABASE}.{SCHEMA_RAW}.TAXI_ZONES tz_pu ON PULOCATIONID = tz_pu.LOCATIONID
LEFT JOIN {SNOWFLAKE_DATABASE}.{SCHEMA_RAW}.TAXI_ZONES tz_do ON DOLOCATIONID = tz_do.LOCATIONID
LEFT JOIN {SNOWFLAKE_DATABASE}.{SCHEMA_RAW}.DIM_VENDOR v ON VENDORID = v.VENDOR_ID
LEFT JOIN {SNOWFLAKE_DATABASE}.{SCHEMA_RAW}.DIM_PAYMENT_TYPE pt ON PAYMENT_TYPE  = pt.PAYMENT_TYPE_ID
LEFT JOIN {SNOWFLAKE_DATABASE}.{SCHEMA_RAW}.DIM_RATE_CODE rc ON RATECODEID = rc.RATE_CODE_ID
LEFT JOIN {SNOWFLAKE_DATABASE}.{SCHEMA_RAW}.DIM_TRIP_TYPE tt ON CAST(TRIP_TYPE AS INTEGER) = tt.TRIP_TYPE_ID
"""
print("Ejecutando UNION ALL con enriquecimiento...")
cursor.execute(sql)
conn.close()
print(f"\n INT_TRIPS_ENRICHED creada en {SCHEMA_RAW}")


Creando INT_TRIPS_ENRICHED...
Ejecutando UNION ALL con enriquecimiento...

 INT_TRIPS_ENRICHED creada en RAW


## Verificaciones

Realiza un control de calidad donde cuenta el total de filas, también cuenta por servicio y por año y cuenta nulos en columnas clave


Básicamente verifica que los JOINs funcionaron bien. 

In [24]:
print("Verificacion de la tabla INT_TRIPS_ENRICHED")


conn = snowflake.connector.connect(
    account=SNOWFLAKE_ACCOUNT, user=SNOWFLAKE_USER,
    password=SNOWFLAKE_PASSWORD, database=SNOWFLAKE_DATABASE,
    schema=SCHEMA_RAW, warehouse=SNOWFLAKE_WAREHOUSE, role=SNOWFLAKE_ROLE,
)
cursor = conn.cursor()

# Total de filas 
cursor.execute(f"SELECT COUNT(*) FROM {SCHEMA_RAW}.INT_TRIPS_ENRICHED")
total = cursor.fetchone()[0]
print(f"\nTotal filas: {total:,}")

# Por servicio 
cursor.execute(f"""
    SELECT SERVICE_TYPE, COUNT(*) as TRIPS
    FROM {SCHEMA_RAW}.INT_TRIPS_ENRICHED
    GROUP BY SERVICE_TYPE
    ORDER BY SERVICE_TYPE
""")
print("\nPor servicio:")
for row in cursor:
    print(f"  {row[0]}: {row[1]:,}")

# Por año 
cursor.execute(f"""
    SELECT SOURCE_YEAR, COUNT(*) as TRIPS
    FROM {SCHEMA_RAW}.INT_TRIPS_ENRICHED
    GROUP BY SOURCE_YEAR
    ORDER BY SOURCE_YEAR
""")
print("\nPor año:")
for row in cursor:
    print(f"  {row[0]}: {row[1]:,}")

# Nulos en columnas clave 
cursor.execute(f"""
    SELECT
        SUM(CASE WHEN PICKUP_DATETIME  IS NULL THEN 1 ELSE 0 END) as null_pickup,
        SUM(CASE WHEN DROPOFF_DATETIME IS NULL THEN 1 ELSE 0 END) as null_dropoff,
        SUM(CASE WHEN PU_LOCATION_ID   IS NULL THEN 1 ELSE 0 END) as null_pu,
        SUM(CASE WHEN DO_LOCATION_ID   IS NULL THEN 1 ELSE 0 END) as null_do,
        SUM(CASE WHEN PU_ZONE          IS NULL THEN 1 ELSE 0 END) as null_pu_zone,
        SUM(CASE WHEN VENDOR_NAME      IS NULL THEN 1 ELSE 0 END) as null_vendor
    FROM {SCHEMA_RAW}.INT_TRIPS_ENRICHED
""")
row = cursor.fetchone()
print("\nNulos en columnas clave:")
print(f"  PICKUP_DATETIME  : {row[0]:,}")
print(f"  DROPOFF_DATETIME : {row[1]:,}")
print(f"  PU_LOCATION_ID   : {row[2]:,}")
print(f"  DO_LOCATION_ID   : {row[3]:,}")
print(f"  PU_ZONE          : {row[4]:,}")
print(f"  VENDOR_NAME      : {row[5]:,}")

conn.close()

print("\n Verificaciones completas de enriquecimiento y unificación")


Verificacion de la tabla INT_TRIPS_ENRICHED

Total filas: 869,792,294

Por servicio:
  green: 68,239,054
  yellow: 801,553,240

Por año:
  2015: 165,272,996
  2016: 147,517,346
  2017: 125,237,386
  2018: 111,771,105
  2019: 90,899,429
  2020: 26,383,268
  2021: 31,973,063
  2022: 40,496,500
  2023: 39,097,286
  2024: 41,829,938
  2025: 49,313,977

Nulos en columnas clave:
  PICKUP_DATETIME  : 0
  DROPOFF_DATETIME : 0
  PU_LOCATION_ID   : 0
  DO_LOCATION_ID   : 0
  PU_ZONE          : 0
  VENDOR_NAME      : 0

 Verificaciones completas de enriquecimiento y unificación
